# SP-01 Dengue Intelligence · Notebook reproducible

Flujo completo de experimentación contra la API de la plataforma: autenticación, verificación del
estado, entrenamiento de un modelo, seguimiento del progreso, promoción, pronóstico, explicabilidad
y comparación estadística.

> **Datos sintéticos.** Este notebook funciona contra la instancia de demostración. Sustituya los
> datos por registros autorizados de MINSA/CDC antes de obtener conclusiones científicas.

Ejecute las celdas en orden. Requiere que la plataforma esté en marcha (`python run_api.py`).

In [ ]:
!pip install -q httpx pandas matplotlib

In [ ]:
import time
import httpx
import pandas as pd

BASE = "http://127.0.0.1:8000/api/v1"
EMAIL = "admin@dengai.unt.edu.pe"
PASSWORD = "Admin#Dengue2025"

client = httpx.Client(base_url=BASE, timeout=600.0)
tokens = client.post("/auth/login", json={"email": EMAIL, "password": PASSWORD}).json()
client.headers["Authorization"] = f"Bearer {tokens['access_token']}"
print("Autenticado. Cambio de contraseña requerido:", tokens["must_change_password"])

In [ ]:
health = client.get("/health").json()
print("Entorno:", health["environment"], "| versión:", health["version"])
print("PyTorch:", health["ml_backend"]["torch"], "\nXGBoost:", health["ml_backend"]["xgboost"])
for limitation in health["limitations"]:
    print("-", limitation)

## 1. Situación epidemiológica actual

In [ ]:
dashboard = client.get("/dashboard/overview", params={"weeks": 104}).json()
print("Datos sintéticos:", dashboard["is_synthetic_data"])
for kpi in dashboard["kpis"]:
    print(f"  {kpi['label']}: {kpi['value']} {kpi.get('unit', '')}")

series = pd.DataFrame(dashboard["weekly_series"])
series["week"] = pd.to_datetime(series["week"])
series.tail()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(series["week"], series["cases"], label="Casos semanales", color="#0b5d6e")
ax.plot(series["week"], series["cases"].rolling(4).mean(), label="Media móvil 4 sem.", color="#17a2b8")
ax.set_title("Casos confirmados por semana epidemiológica (datos sintéticos)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Entrenamiento de un modelo

In [ ]:
ALGORITHM = "xgboost"   # xgboost | lstm | convlstm | cnn_lstm | convlstm_attention
HORIZONS = [1]
TRIALS = 10

created = client.post("/ml/experiments", json={
    "algorithm": ALGORITHM, "n_trials": TRIALS, "horizons": HORIZONS,
    "validation_strategy": "walk_forward",
})
created.raise_for_status()
experiment_id = created.json()["id"]
print("Experimento", experiment_id, "en cola")

In [ ]:
status = ""
while True:
    detail = client.get(f"/ml/experiments/{experiment_id}").json()
    if detail["status"] != status:
        print(f"{detail['status']:10s} {detail['stage']:40s} {detail['progress']:.0%}")
        status = detail["status"]
    if detail["status"] in ("COMPLETED", "FAILED"):
        break
    time.sleep(3)

if detail["status"] == "FAILED":
    raise RuntimeError(detail["error_message"])
print("Duración:", detail["duration_seconds"], "s")

## 3. Evaluación: métricas, intervalos y particiones

In [ ]:
metrics_table = []
for horizon, payload in detail["metrics"].items():
    row = {"horizonte": horizon, **{k: v for k, v in payload["metrics"].items() if v is not None}}
    metrics_table.append(row)
pd.DataFrame(metrics_table).set_index("horizonte")

In [ ]:
notes = detail["metrics"][str(HORIZONS[0])]["notes"]
print("Particiones (sin fuga de información):")
for key in ("train_end", "val_range", "test_start", "strategy"):
    print(f"  {key}: {notes.get(key)}")
print("Mejores hiperparámetros:", detail["metrics"][str(HORIZONS[0])]["best_params"])

## 4. Promoción y pronóstico

In [ ]:
models = client.get("/ml/models").json()
mine = [m for m in models if detail["run_id"][:8] in m["version"]]
if mine:
    print(client.post(f"/ml/models/{mine[0]['id']}/promote").json()["message"])

forecast = client.post("/forecasts/generate", json={"horizons": HORIZONS}).json()
frame = pd.DataFrame(forecast["predictions"])
frame[["district_name", "target_year", "target_week", "horizon_weeks", "predicted_cases",
       "lower_bound", "upper_bound", "incidence_per_100k", "risk_level"]].sort_values(
    "predicted_cases", ascending=False).head(10)

## 5. Explicabilidad (SHAP)

In [ ]:
district = frame.sort_values("predicted_cases", ascending=False).iloc[0]["district_code"]
explanation = client.get(f"/ml/explain/{district}", params={"horizon": HORIZONS[0]}).json()
print("Método:", explanation["method"], "| Predicción:", explanation["prediction"])
print(explanation["narrative"])
pd.DataFrame(explanation["factors"])[["label", "contribution", "direction", "value"]]

## 6. Comparación estadística entre modelos

In [ ]:
comparison = client.get("/ml/compare", params={"horizon": HORIZONS[0]})
if comparison.status_code == 200:
    body = comparison.json()
    print("Ganador:", body["winner"], "|", body["winner_basis"])
    display(pd.DataFrame(body["ranking"]))
    display(pd.DataFrame(body["tests"])[["test_name", "comparison", "p_value", "significant"]])
else:
    print("Se requieren al menos dos modelos entrenados:", comparison.json())

## 7. Alertas y reporte

In [ ]:
alerts = client.post("/alerts/generate", json={"horizons": HORIZONS, "min_level": "MODERATE"}).json()
print("Alertas generadas o vigentes:", alerts["created"])

report = client.post("/reports", json={
    "report_type": "resumen_epidemiologico", "format": "pdf",
}).json()
print("Reporte:", report["report_key"], report["file_size"], "bytes")

pdf = client.get(f"/reports/{report['id']}/download")
with open("resumen_epidemiologico.pdf", "wb") as handle:
    handle.write(pdf.content)
print("PDF guardado en resumen_epidemiologico.pdf")

## 8. Trazabilidad

La verificación de la cadena de auditoría demuestra que los eventos registrados no fueron
alterados. Si la verificación falla, la plataforma indica el primer evento inconsistente.

In [ ]:
verification = client.get("/audit/verify").json()
print(verification["message"])
print("Eventos recientes:")
for event in client.get("/audit", params={"size": 5}).json()["items"]:
    print(f"  {event['occurred_at'][:19]} {event['event_type']:16s} {event['module']:6s} {event['description'][:70]}")